### GridWorld 구축
환경 파라미터를 정의

In [5]:
import numpy as np

In [6]:
grid_size = (3, 4)
blocked_cell = (1, 1)
baseline_reward = -0.02
absorbing_cells = {(0,3): 1, (1,3): -1}

actions = ['L', 'U', 'R', 'D']

num_actions = len(actions)

probs = [.1, .8, .1, 0]

In [7]:
# 1차원 2차원 표현 간에 전환을 자주해야 함 -> 2개의 helper 함수를 정의
# 상태는 1차원, 셀은 상응하는 2차원 좌표
to_1d = lambda x: np.ravel_multi_index(x, grid_size)
to_2d = lambda x: np.unravel_index(x, grid_size)

In [8]:
# 데이터 포인트를 미리 계산해서 코드를 더욱 간결하게 만듦
num_states = np.product(grid_size)
cells = list(np.ndindex(grid_size))
states = list(range(len(cells)))
cell_state = dict(zip(cells, states))
state_cell = dict(zip(states, cells))
absorbing_states = {to_1d(s):r for s, r in absorbing_cells.items()}
blocked_state = to_1d(blocked_cell)

In [10]:
# 각 상태에 대한 보상을 저장
state_rewards = np.full(num_states, baseline_reward)
state_rewards[blocked_state] = 0
for state, reward in absorbing_states.items():
    state_rewards[state] = reward

state_rewards

array([-0.02, -0.02, -0.02,  1.  , -0.02,  0.  , -0.02, -1.  , -0.02,
       -0.02, -0.02, -0.02])

In [15]:
action_outcomes = {}
for i, action in enumerate(actions):
    probs_ = dict(zip([actions[j % 4] for j in range(i, num_actions + i)],probs))
    action_outcomes[actions[(i+1) % 4]] = probs_

action_outcomes

{'U': {'L': 0.1, 'U': 0.8, 'R': 0.1, 'D': 0},
 'R': {'U': 0.1, 'R': 0.8, 'D': 0.1, 'L': 0},
 'D': {'R': 0.1, 'D': 0.8, 'L': 0.1, 'U': 0},
 'L': {'D': 0.1, 'L': 0.8, 'U': 0.1, 'R': 0}}

### 전이행렬 계산
각 이전 상태와 행동 A에 대해 특정 상태 S에 종학할 확률 P(s'|s,a)를 정의함
pymdptoolbox 예시하고, 전이와 보상을 지정하고자 가능한 공식 중 하나를 사용한다.
전이확률 모두에 대해서 A*S*S의 차원을 가진 넘파이 배열을 만든다.

In [16]:
def get_new_cell(state, move):
    cell = to_2d(state)
    if actions[move] == 'U':
      return cell[0] - 1, cell[1]
    elif actions[move] == 'D':
      return cell[0] + 1, cell[1]
    elif actions[move] == 'R':
      return cell[0], cell[1] + 1
    elif actions[move] == 'L':
      return cell[0], cell[1] - 1

다음 함수는 state, action, outcome을 지정하는 인수를 사용해 전이확률과 보상을 채운다.

In [ ]:
def update_transition_and_rewards(state, action, outcome):
    if state in absorbing_states.keys() or state == blocked_state:
        transitions[action, state, state] = 1
    else:
        new_cell = get_new_cell(state, outcome)
        p = action_outcomes[actions[action]][actions[outcome]]
        if new_cell not in cells or new_cell == blocked_cell:
            transitions[action, state, state] += p
            rewards[action, state, state] = baseline_rewards